In [2]:
# Configura o Bedrock e a Lambda
import boto3
import json

bedrock = boto3.client(service_name='bedrock-runtime')
lambda_client = boto3.client("lambda")
MODEL_ID = "amazon.nova-micro-v1:0"

# Define a ferramenta de cálculo
math_tool = {
    "toolSpec": {
        "name": "calculateNumbers",
        "description": "Realiza operações aritméticas básicas",
        "inputSchema": {
            "json": {
                "type": "object",
                "properties": {
                    "operation": {"type": "string"},
                    "num1": {"type": "number"},
                    "num2": {"type": "number"}
                },
                "required": ["operation", "num1", "num2"]
            }
        }
    }
}

# Função que aciona o serviço de cálculo na Lambda
def execute_calculation(input_data):
    response = lambda_client.invoke(
        FunctionName="math-function",  
        InvocationType="RequestResponse",
        Payload=json.dumps(input_data)
    )
    response_payload = response["Payload"].read()
    calculation_result = json.loads(response_payload)
    response_body = calculation_result.get("body", "{}")
    return json.loads(response_body) if isinstance(response_body, str) else response_body

# Mensagem inicial do usuário
user_input = {
    "role": "user",
    "content": [{"text": "Por favor, subtraia 60 de 100"}]
}

# Define as instruções de sistema
system_instructions = [
    {"text": """
    Você é um assistente virtual capaz de realizar operações aritméticas básicas: add (somar), subtract (subtrair), multiply (multiplicar) e divide (dividir).
    Ao usar a ferramenta, informe a operação com o nome em inglês (add, subtract, multiply ou divide).
    Se o usuário não especificar uma operação, peça mais detalhes.
    """}
]

# Primeira interação com o modelo
first_interaction = bedrock.converse(
    modelId=MODEL_ID,
    system=system_instructions,
    messages=[user_input],
    toolConfig={
        "tools": [math_tool],
        "toolChoice": {"auto": {}}
    },
    inferenceConfig={"temperature": 0.7}
)

# Processa a resposta do assistente para verificar se a ferramenta é necessária
assistant_reply = first_interaction["output"]["message"]
message_parts = assistant_reply["content"]
tool_request_block = next((part for part in message_parts if "toolUse" in part), None)

if not tool_request_block:
    print("=== Resposta direta do assistente ===")
    print(message_parts[0]["text"])
else:
    tool_request = tool_request_block["toolUse"]
    tool_input_data = tool_request["input"]
    tool_id = tool_request["toolUseId"]
    print(tool_request_block)
    print(f"→ O assistente acionou a ferramenta calculateNumbers com a entrada: {tool_input_data}")

    # Executa a ferramenta solicitada
    tool_result = execute_calculation(tool_input_data)
    print(f"← Saída da função Lambda: {tool_result}")

    # Cria uma resposta com base na saída da ferramenta
    try:
        result_summary = f"O resultado do cálculo é {tool_result['result']}."
    except Exception as e:
        result_summary = f"Ops! Houve um erro no cálculo. ({str(e)})"

    # Gera a mensagem com o resultado da ferramenta
    tool_response_msg = {
        "role": "user",
        "content": [
            {
                "toolResult": {
                    "toolUseId": tool_id,
                    "content": [{"text": result_summary}]
                }
            }
        ]
    }

    # Envia o resultado da ferramenta de volta ao modelo
    final_output = bedrock.converse(
        modelId=MODEL_ID,
        messages=[user_input, assistant_reply, tool_response_msg],
        toolConfig={  
            "tools": [math_tool],
            "toolChoice": {"auto": {}}
        },
        inferenceConfig={"temperature": 0.7}
    )

    # Exibe a resposta final do assistente
    final_message = final_output["output"]["message"]["content"][0]["text"]
    print("\n=== Resposta final do assistente ===")
    print(final_message)

{'toolUse': {'toolUseId': 'tooluse_V1eE0kDjLFlSVc3qJ9UekT', 'name': 'calculateNumbers', 'input': {'num1': 100, 'operation': 'subtract', 'num2': 60}}}
→ Assistant triggered tool: calculateNumbers with input: {'num1': 100, 'operation': 'subtract', 'num2': 60}
← Lambda Function output: {'result': 40.0}

=== Final Assistant Response ===
The outcome of subtracting 60 from 100 is 40.0.
